# Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path
import torch

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from hydra.utils import instantiate # noqa: E402
from src.utils.notebook_setup import init_nlp_notebook # noqa: E402

cfg = init_nlp_notebook()
device = "cuda" if torch.cuda.is_available() else "cpu"

# Classification Revue (optional)

In [ ]:
# Эта ячейка актуальна, если ты обучал классификатор (энкодер)
ckpt_path = cfg.get("ckpt_path") # Путь к .ckpt файлу, например "models/checkpoints/best-epoch=05.ckpt"

if ckpt_path and ckpt_path.endswith(".ckpt"):
    print(f"Запуск оценки Lightning-модели из: {ckpt_path}")
    
    tokenizer = instantiate(cfg.model.tokenizer).build()
    
    # Собираем DataModule для тестовой выборки
    datamodule = instantiate(cfg.datamodule, tokenizer=tokenizer)
    datamodule.prepare_data()
    datamodule.setup(stage="test")
    
    # Собираем базовую модель
    base_model = instantiate(cfg.model.builder, tokenizer=tokenizer).build()
    model_module = instantiate(cfg.model_module, model=base_model)
    
    # Инициализируем Trainer (только для теста, без логгеров типа wandb)
    trainer = instantiate(cfg.trainer, logger=False)
    
    # Вся магия торчметрикс произойдет здесь!
    test_results = trainer.test(model=model_module, datamodule=datamodule, ckpt_path=ckpt_path)
    print("\nФинальные метрики:", test_results)
else:
    print("Пропуск ячейки: Чекпоинт не передан или не является файлом .ckpt")

# Gen model init (peft/lora)

In [ ]:
# Переопределяем паддинг для пакетной генерации
cfg.model.tokenizer.padding_side = "left"
tokenizer = instantiate(cfg.model.tokenizer).build()

# 1. Загружаем БАЗОВУЮ архитектуру
print(f"Загрузка базовой модели: {cfg.model.builder.model_name_or_path}")
model = instantiate(cfg.model.builder, tokenizer=tokenizer).build()

# 2. Подгружаем обученные веса
lora_path = cfg.get("ckpt_path") # Например, "models/lora_adapters/run_01"

if lora_path and os.path.exists(os.path.join(lora_path, "adapter_config.json")):
    print("Обнаружен PEFT/LoRA адаптер. Оборачиваем модель...")
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, lora_path)
    model.to(device)
    model.eval()
else:
    print("LoRA адаптер не найден. Оценка будет проведена на базовой модели.")

# Инициализация генератора
generator = instantiate(cfg.model.generation, model=model, tokenizer=tokenizer)

# Torchmetrics Text

In [ ]:
from torchmetrics.text.rouge import ROUGEScore
from src.core.models.promts import PromptManager
from tqdm.auto import tqdm

# Инициализируем метрику
rouge_metric = ROUGEScore()

# Предположим, мы оцениваем на валидационной выборке
datamodule = instantiate(cfg.data.module, tokenizer=tokenizer)
datamodule.setup(stage="validate")

text_col = cfg.data.text_column
target_col = cfg.data.get("target_column", "target")

eval_samples = datamodule.val_dataset.select(range(min(100, len(datamodule.val_dataset))))
prompts = [PromptManager.build_simple_prompt(row[text_col]) for row in eval_samples]
references = [row[target_col] for row in eval_samples]

batch_size = 8
all_preds = []

print(f"Запуск пакетной генерации ({len(prompts)} сэмплов)...")
for i in tqdm(range(0, len(prompts), batch_size)):
    batch_prompts = prompts[i : i + batch_size]
    
    # Твой генератор сам обрабатывает батчи!
    preds = generator.generate(batch_prompts, max_new_tokens=50)
    all_preds.extend(preds)

# Считаем метрики (torchmetrics ожидает списки строк)
rouge_metric.update(all_preds, references)
results = rouge_metric.compute()

print("\n--- Финальные метрики генерации ---")
for metric_name, tensor_val in results.items():
    # Извлекаем FMeasure (fmeasure) из тензора
    print(f"{metric_name}: {tensor_val.item():.4f}")

# Visual

In [ ]:
import random

print("--- Случайные примеры для визуального контроля ---")
indices = random.sample(range(len(all_preds)), min(3, len(all_preds)))

for idx in indices:
    print(f"\n[ПРОМПТ]:\n{prompts[idx]}")
    print(f"\n[ОЖИДАЛОСЬ]:\n{references[idx]}")
    print(f"\n[ПРЕДСКАЗАНО]:\n{all_preds[idx]}")
    print("-" * 50)